In [ ]:
import os
from data_processing.scripts.post_processing_checks import verify_cube_chunks, find_good_indices
from data_processing.scripts.plot_helpers_new import find_cloud_free_indices, plot_statistical_outliers, plot_acquisition_timelines, plot_rgb, plot_acquisition_timelines_filtered, plot_nan_distribution, plot_spatial_nan_frequency, plot_variable_analysis
from data_processing.scripts.sentinel_2_processing import get_s2_quality_masks
import xarray as xr
from data_processing.scripts.plot_helpers import plot_landcover
from data_processing.scripts.sentinel_2_processing import *
from data_processing.scripts.cube_processing  import add_event_metadata
import matplotlib.pyplot as plt
import random
import s3fs
import fsspec
import pandas as pd
from data_processing.cube_processing import process_era5, load_global_stats
from data_processing.scripts.era_5_processing import aggregate_era5_metrics_new
from data_processing.scripts.normalize_and_clip import normalize_era5
from dotenv import load_dotenv, find_dotenv

In [ ]:
load_dotenv(find_dotenv())
CUBE_DIR = os.getenv("OUTPUT_DIR")
era5_dir = os.path.join(CUBE_DIR, "era5")
era5_list = os.listdir(era5_dir)
print(len(era5_list))

In [ ]:
cube_list = os.listdir(CUBE_DIR)
cube_list = [file for file in cube_list if file[0] == "2"]
print(len(cube_list))

In [ ]:
stats_path = os.path.join(CUBE_DIR, "global_era5_stats.json")
global_era5_stats = load_global_stats(stats_path)
global_era5_stats

In [ ]:
cube_list

In [ ]:
# cube_id = "2021-0561-ESP"
cube_id = cube_list[0].split(".zarr")[0]
era5_daily = xr.open_zarr(os.path.join(era5_dir, f"{cube_id}_era5.zarr" ))
ds = xr.open_zarr(os.path.join(CUBE_DIR, f"{cube_id}.zarr" ))
# ds.pei_90_mean.plot()
plt.figure(figsize=(5, 4))
era5_daily.t2m.plot()
plt.show()

In [ ]:
vars_list = list(era5_daily.data_vars)
era5_resampled = aggregate_era5_metrics_new(era5_daily, ds, vars_list)

In [ ]:
era5_resampled.t2m_mean.plot()

In [ ]:
era5_resampled = normalize_era5(era5_resampled, global_era5_stats)

In [ ]:
era5_resampled.tp_rollingmax_mean.plot()

In [ ]:
for var, s in global_era5_stats.items():
    if var in era5_resampled.data_vars:
        print(var)
        print("Original_value: ", era5_resampled[var].isel(time_sentinel_2_l2a))
        print("Mean: ", s["mean"])
        print("Std: ", s["std"])
        normalized = (era5_resampled[var] - s["mean"]) / s["std"]

        print("Normalized: ", normalized.isel(time_sentinel_2_l2a))

        # Assert: Check if we are creating infinite values
        # assert not np.isinf(
        #     normalized
        # ).any(), f"Normalization produced Inf values in {var}"

        # ds[var] = normalized.astype("float32")

In [ ]:
for c in era5_list:
    ds = xr.open_zarr(os.path.join(era5_dir, c ))
    # ds.pei_90_mean.plot()
    plt.figure(figsize=(5, 4))
    ds.t2m.plot()
    plt.show()

In [ ]:
cubes = {}
for i, cube in enumerate(cube_list):
    ds = xr.open_zarr(os.path.join(cube_dir, cube_list[i]))
    cube_id = ds.attrs["cube_id"]
    print(cube_id)
    cubes[cube_id] = ds

In [ ]:
global_era5_stats = process_era5(cubes, era5_cubes, OUTPUT_DIR)

In [ ]:
output_dir = "/scratch/sloeblein_new"
from scripts.era_5_processing import (
    subset_era5_spatial,
    subset_era5_time,
    aggregate_era5_metrics_new,
    check_time_alignment)

In [ ]:
from scripts.normalize_and_clip import (
    normalize_dem,
    calculate_global_era5_stats,
    normalize_era5,
)


In [ ]:
ds = xr.open_zarr(os.path.join(cube_dir, cube_list[2]))
cube_id = ds.attrs["cube_id"]
cube_id

In [ ]:
ds.pei_90_mean.plot()

In [ ]:
from scripts.era_5_processing import (
    subset_era5_spatial,
    subset_era5_time,
    aggregate_era5_metrics_new,
    check_time_alignment,
)
ERA5_PATH = os.getenv("ERA5_DATA_PATH")
pei_cube_name = "PEICube_era5land.zarr"
t2_cube_name = "t2_ERA5land.zarr"
tp_cube_name = "tp_ERA5land.zarr"
pei_cube = xr.open_zarr(os.path.join(ERA5_PATH, pei_cube_name), consolidated=False)
t2_cube = xr.open_zarr(os.path.join(ERA5_PATH, t2_cube_name), consolidated=False)
tp_cube = xr.open_zarr(os.path.join(ERA5_PATH, tp_cube_name), consolidated=False)
era5_cubes = [pei_cube, t2_cube, tp_cube]

In [ ]:
t2_cube.t2m

In [ ]:
cube_features= []

for i, era5_cube_raw in enumerate(era5_cubes):
    print(i)
    # 1. Temporal subset
    tmp = subset_era5_time(era5_cube_raw, ds)
    print(ds.time_sentinel_2_l2a[0].values)
    print(tmp.Ti[0].values)

    tmp, _ = subset_era5_spatial(tmp, ds, plot_check=False)

    vars_list = list(tmp.data_vars)
    tmp = aggregate_era5_metrics_new(tmp, ds, vars_list)

    tmp = tmp.squeeze(dim=["latitude", "longitude"], drop=True)
    cube_features.append(tmp)

In [ ]:
# Direkt vor era5_merged = xr.merge(cube_features) einfügen:
for i, feat in enumerate(cube_features):
    print(f"Feature {i} Zeit-Länge: {len(feat.time_sentinel_2_l2a)}")
    # Zeigt die ersten 3 Zeitstempel zur Kontrolle
    print(f"Erste 3 Timesteps:\n{feat.time_sentinel_2_l2a.values[:3]}")

In [ ]:
era5_merged = xr.merge(cube_features)

In [ ]:
era5_path = f"{era5_dir}/{key}_era5.zarr"
era5_merged.to_zarr(era5_path, mode="w", consolidated=True)

In [ ]:
# 1. Haben beide exakt die gleiche Anzahl an Zeitschritten?
print(f"S2 Zeitlänge: {len(ds.time_sentinel_2_l2a)}")
print(f"ERA5 Zeitlänge: {len(era5_merged.time_sentinel_2_l2a)}")

# 2. Gibt es NaNs direkt nach dem reindex?
for v in era5_merged.data_vars:
    nan_count = era5_merged[v].isnull().sum().compute().item()
    if nan_count > 0:
        print(f"⚠️ Variable {v} hat {nan_count} NaNs nach dem Reindex!")

# 3. Test-Merge
try:
    test_merge = xr.merge([ds, era5_merged], join="exact")
    print("✅ Merge erfolgreich ohne neue Zeitstempel zu erzeugen.")
except ValueError as e:
    print(f"❌ Merge-Fehler: {e}")

In [ ]:
ds.tp_dailymax_max.plot()

In [ ]:
era5_merged.tp_dailymax_max.plot()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xarray as xr
from typing import Tuple, Optional, Union, List
import pandas as pd
import numpy as np

def plot_rgb(
    ds: xr.Dataset,
    timestep: int,
    time_dim: str = "time_sentinel_2_l2a",
    bands: Tuple[str, str, str] = ("B04", "B03", "B02"),
    stretch_pct: Tuple[int, int] = (2, 98),
    figsize: Tuple[int, int] = (10, 6),
    cloud_comp: bool = False,
    title_prefix: Optional[str] = "Timestep",
) -> Union[None, str]:
    """
    Plots a static true-color (RGB) composite for a specific timestep
    from a Sentinel-2 xarray.Dataset. Optionally plots a cloud mask comparison.

    Parameters
    ----------
    ds : xarray.Dataset
        Must have a time coordinate `time_dim` and DataArrays for the three bands.
    timestep : int
        The time index to plot (must be valid for ds[time_dim]).
    time_dim : str
        Name of the time dimension (default 'time_sentinel_2_l2a').
    bands : tuple of str
        The (red, green, blue) band names in ds (default ('B04','B03','B02')).
    stretch_pct : tuple of int
        Percentiles for contrast stretch (default 2nd to 98th).
    figsize : tuple
        Matplotlib figure size.
    cloud_comp : bool
        If True, plots the RGB image and the cloud mask comparison side-by-side.
    title_prefix : str or None
        Prefix for the figure title (used by interactive function).

    Returns
    -------
    None or str
        None on success, or an error message string if the timestep is invalid.
    """

    # Helper function to percentile‐stretch one (y,x) slice
    def stretch_arr(band):
        arr = band.compute().astype("float32")
        p2, p98 = np.percentile(arr, stretch_pct)
        return np.clip((arr - p2) / (p98 - p2), 0, 1)

    # 1. Input Validation
    if not 0 <= timestep < len(ds.coords[time_dim]):
        return f"Timeindex {timestep} out of range for dimension '{time_dim}' (size {len(ds.coords[time_dim])})."

    # 2. Subset data for the given timestep
    subset = ds.isel({time_dim: timestep})
    time_value_raw = subset[time_dim].item()

    time_formatted = pd.to_datetime(time_value_raw).strftime("%Y-%m-%d")

    try:
        r, g, b = subset[bands[0]], subset[bands[1]], subset[bands[2]]
    except KeyError as e:
        return f"One or more bands not found in dataset: {e}"

    # 3. Create RGB array (stretched)
    rgb = np.dstack(
        [stretch_arr(r).values, stretch_arr(g).values, stretch_arr(b).values]
    )

    # 4. Plotting Logic

    if cloud_comp:
        # --- Cloud Comparison Logic ---
        try:
            scl = subset["SCL"]
            cloud_mask = subset["cloud_mask"]
        except KeyError as e:
            return f"Required variable for cloud comparison missing: {e}. Set cloud_comp=False or check dataset."

        # Binary masks
        is_cloud_scl = scl.isin([3, 8, 9, 10])  # cloud classes SCL
        is_external_cloud = cloud_mask.isin([1, 2, 3])  # cloud classes cloud_mask

        # Calculate difference and agreement masks for the overlay
        scl_only_mask = (
            (is_cloud_scl & ~is_external_cloud).astype(float).values.squeeze()
        )
        external_only_mask = (
            (is_external_cloud & ~is_cloud_scl).astype(float).values.squeeze()
        )
        agreement_mask = (
            (is_external_cloud & is_cloud_scl).astype(float).values.squeeze()
        )

        # Create the RGB overlay (Red=SCL Only, Green=External Only, Blue=Agreement)
        overlay = np.dstack(
            [
                scl_only_mask,
                external_only_mask,
                agreement_mask,
            ]
        )

        # Plotting side by side (2 plots)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

        # Left Plot: RGB ONLY
        ax1.imshow(rgb)
        ax1.set_title("True Color (RGB) Image", fontsize=12)
        ax1.axis("off")

        # Right Plot: RGB + Mask Comparison Overlay
        ax2.imshow(rgb)
        ax2.imshow(overlay, alpha=0.5)

        ax2.set_title(
            "Mask Comparison\n" "Red: SCL Only | Green: External Only | Blue: Both",
            fontsize=12,
        )
        ax2.axis("off")

        fig.suptitle(f"{title_prefix}: {time_formatted}", fontsize=14, y=1.02)

    else:
        # Plotting RGB ONLY (1 plot)
        fig, ax1 = plt.subplots(1, 1, figsize=figsize, constrained_layout=True)

        ax1.imshow(rgb)
        ax1.set_title(
            f"True Color (RGB) Image\n{title_prefix}: {time_formatted}", fontsize=14
        )
        ax1.axis("off")

    # plt.show()
    return fig

In [ ]:
def find_cloud_free_indices(ds, threshold=0.99):
    """
    Finds indices of Sentinel 2 timesteps, that are almost cloud free.
    Based on the "mask_phys_strict" mask
    """
    # 1. Calculates mean over x and y per timestep
    quality_means = ds.mask_phys_strict.mean(dim=("x", "y"))

    # 2. Find indices which meet criteria
    indices = np.where(quality_means > threshold)[0]

    # 3. Calculate amount of indices
    count = len(indices)

    print(f"Found cloudfree timesteps (> {threshold*100}%): {count}")

    return indices.tolist()
    
indices_rgb = find_cloud_free_indices(ds_plot)
if indices_rgb is not None and len(indices_rgb) > 0:
    idx = int(indices_rgb[0]) 
    fig = plot_rgb(ds_plot, idx)

In [ ]:
S3_BASE_URL = os.getenv("S3_BASE_URL")
BUCKET_NAME = os.getenv("BUCKET_NAME")
S3_ENDPOINT = os.getenv("S3_ENDPOINT")

fs = s3fs.S3FileSystem(
    anon=True, client_kwargs={"endpoint_url": S3_ENDPOINT}
)
url = f"{S3_BASE_URL}{BUCKET_NAME}/DC__{cube_id}.zarr"
mapper = fsspec.get_mapper(url)
ds_plot = xr.open_zarr(mapper, consolidated=True)
ds_plot = get_s2_quality_masks(ds_plot)
indices_rgb = find_cloud_free_indices(ds_plot)
fig = plot_rgb(ds_plot, indices_rgb[0])

In [ ]:
verify_cube_chunks(os.path.join(cube_dir, cube_list[0]))

In [ ]:
S2_VARS = ["NDVI", "kNDVI", "CIRE", "IRECI", "NDWI", "NDMI", "NIRv"]
S1_VARS = ["vv", "vh"]
STATIC_VARS = ["ESA_LC", "COP_DEM", "is_veg"]
indices = find_good_indices(ds)   

In [ ]:
for var in S2_VARS:
    print("#"*10, {var}, "#"*10)
    plt.figure(figsize=(8, 6))
    
    ds[var].isel(time_sentinel_2_l2a=indices[0]).plot(robust=True, cmap="viridis")
    
    plt.title(f"Variable: {var} at index {indices[0]}")
    plt.show()

In [ ]:
# 1. Calculate percentage of NaNs per timesteps 
nan_ratio = ds[S1_VARS[0]].isnull().mean(dim=["x", "y"]).compute()

# 2. Find all indices where nan_ratio <0.1
valid_indices = np.where(nan_ratio < 0.1)[0]

if len(valid_indices) > 0:
    # 3. Select random valid index
    idx = int(random.choice(valid_indices))
    print(f"Selected Index: {idx} (NaN-Proportion: {nan_ratio[idx].values:.2%})")
else:
    print("No timestep with < 10% NaNs found!")
    # Fallback: Take timestep with fewest nans
    idx = int(nan_ratio.argmin())
    print(f"Using best available timestep: {idx} ({nan_ratio[idx].values:.2%} NaNs)")

for var in S1_VARS:
    print("#"*10, {var}, "#"*10)
    plt.figure(figsize=(8, 6))
    
    ds[var].isel(time_sentinel_2_l2a=idx).plot(robust=True, cmap="viridis")
    
    plt.title(f"Variable: {var} at index {idx}")
    plt.show()

In [ ]:
ds.pei_90_mean.dims == ('time_sentinel_2_l2a',)

In [ ]:
era5_vars = [var for var in ds.data_vars if ds[var].dims == ('time_sentinel_2_l2a',)]
era5_var

In [ ]:
for var in (STATIC_VARS):
    print("#"*10, {var}, "#"*10)
    plt.figure(figsize=(8, 6))

    static_dim = ds[var].dims[0]
    # assert len(ds[var][static_dim]) == 1 | len(ds[var][static_dim]) == 1000
    print("Dimension: ")
    print("Length: ", len(ds[var][static_dim]))

    if len(ds[var][static_dim]) == 1:
        ds[var].isel({static_dim: 0}).plot()

    else:
        ds[var].plot()
        
    plt.title(f"Variable: {var}")
    plt.show()



In [ ]:
# Erstelle ein Grid mit 1 Zeile und 3 Spalten
fig, axes = plt.subplots(1, 3, figsize=(25, 10))

# 1. Linker Plot: RGB (Stelle sicher, dass deine plot_rgb auch ein ax akzeptiert!)
plot_rgb(ds_plot, indices_rgb[0], ax=axes[0])
axes[0].set_title("RGB Snapshot")

# 2. Mittlerer Plot: Landcover
plot_landcover(ds, ax=axes[1])

# 3. Rechter Plot: Vegetation Mask
if "is_veg" in ds:
    ds.is_veg.plot(ax=axes[2], add_colorbar=False)
    axes[2].set_title("Vegetation Mask ")
    axes[2].set_aspect('equal')

axes[0].axis("off")
axes[1].axis("off")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for var in (S2_VARS + S1_VARS):
    print("#"*10, {var}, "#"*10)
    plot_spatial_nan_frequency(ds, var, ds.attrs["precip_end_date"])


In [ ]:
from scripts.interpolation import trim_to_first_s2_acquisition, interpolate_context_only

print("Before: ", len(ds.time_sentinel_2_l2a))
ds_intp = trim_to_first_s2_acquisition(ds)
print("After: ", len(ds_intp.time_sentinel_2_l2a))

vars_to_interpolate = [
    v for v in ds.data_vars 
    if  "time_sentinel_2_l2a" in ds[v].dims
    and not any(suffix in v for suffix in ["_count",  "std", "_mask"])
]
print(vars_to_interpolate)

ds_intp = interpolate_context_only(ds_intp, vars_to_interpolate)

### THINK ABOUT MASK -> ARE THERE NANs?
### WHAT ABOUT COUNT: SHOULD IT BE 0 or 1 then and what bout std
## SHOULD MIN; MAX BE INTERPOLATED

In [ ]:
def plot_nan_distribution(ds_old, ds_new, var_name, cutoff_date):
    """
    Plottet NaN-Verteilung und schreibt die Statistik direkt ins Bild.
    """
    time_dim = "time_sentinel_2_l2a"
    cutoff_dt = pd.to_datetime(cutoff_date)

    # 1. Berechnungen
    nan_perc_old = (ds_old[var_name].isnull().mean(dim=["x", "y"]) * 100).compute()
    nan_perc_new_raw = (ds_new[var_name].isnull().mean(dim=["x", "y"]) * 100).compute()
    
    # Reindex, falls ds_new getrimmt wurde (für korrekte x-Achse)
    nan_perc_new = nan_perc_new_raw.reindex_like(nan_perc_old, fill_value=100)
    
    # Absolute Zahlen für die Statistik-Box
    total_pixels = ds_old[var_name].size
    n_before = int(ds_old[var_name].isnull().sum().compute())
    n_after = int(ds_new[var_name].isnull().sum().compute())
    n_filled = n_before - n_after
    fill_rate = (n_filled / n_before * 100) if n_before > 0 else 0

    # 2. Plotting
    fig, ax = plt.subplots(figsize=(15, 6))
    time_axis = ds_old[time_dim].values

    ax.fill_between(time_axis, nan_perc_old, label="Original NaNs", color="#e74c3c", alpha=0.3)
    ax.fill_between(time_axis, nan_perc_new, label="Post-Interpolation", color="#2ecc71", alpha=0.5)

    ax.axvline(x=cutoff_dt, color="black", linestyle="--", linewidth=2, label="Event Cutoff")

    # --- Die "schöne Darstellung" der Zahlen als Textbox ---
    stats_text = (
        f"📊 STATS: {var_name}\n"
        f"{'-'*20}\n"
        f"Total Pixels:  {total_pixels:,}\n"
        f"NaNs Before:   {n_before:,}\n"
        f"NaNs After:    {n_after:,}\n"
        f"Filled:        {n_filled:,}\n"
        f"Recovery Rate: {fill_rate:.1f}%"
    )
    
    # Positionierung oben rechts oder links, je nach Bedarf
    props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='#34495e')
    ax.text(0.02, 0.95, stats_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='top', family='monospace', bbox=props)

    # Styling
    ax.set_title(f"NaN Distribution & Interpolation Success: {var_name}", fontsize=14, pad=15)
    ax.set_ylabel("NaNs per Timestep (%)")
    ax.set_ylim(-2, 105)
    ax.legend(loc="upper right", frameon=True)
    ax.grid(alpha=0.3, linestyle=':')
    
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
cutoff_date = pd.to_datetime(ds.attrs["precip_end_date"])
plot_nan_distribution(ds, ds_intp, "kNDVI", cutoff_date)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 1. Plot RGB on the left
plot_variable_analysis(
    ds, 
    variables_to_plot=["kNDVI", "kNDVI_min", "kNDVI_max"], 
    plot_ts_mean=True, 
    plot_ts_median=True,
    ax=axes[0],
    title="kNDVI before interpolation"
)

# 2. Plot ERA5 and NDVI Stats on the right
plot_variable_analysis(
    ds_intp, 
    variables_to_plot=["kNDVI", "kNDVI_min", "kNDVI_max"], 
    ax=axes[1],
    title="kNDVI after interpolation"
)

plt.tight_layout()
plt.show()

In [ ]:
era5  = xr.open_zarr("/scratch/sloeblein_new/era5/2019-0607-ZAF_era5.zarr")

In [ ]:
np.unique(era5.pei_180_min.values)

In [ ]:
# Schnellster Check mit räumlichem Mittelwert
era5.t2m_mean.plot(marker='o', figsize=(10, 4))

In [ ]:
era5

In [ ]:
plot_variable_analysis(
    ds, 
    variables_to_plot=["vv_count"], 
    title="kNDVI before interpolation"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 1. Plot RGB on the left
plot_variable_analysis(
    ds, 
    variables_to_plot=["vh", "vh_min", "vh_max"], 
    ax=axes[0],
    title="vh before interpolation"
)

# 2. Plot ERA5 and NDVI Stats on the right
plot_variable_analysis(
    ds_intp, 
    variables_to_plot=["vh", "vh_min", "vh_max"], 
    ax=axes[1],
    title="vh after interpolation"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 1. Plot RGB on the left
plot_variable_analysis(
    ds, 
    variables_to_plot=["CIRE", "CIRE_min", "CIRE_max"], 
    plot_ts_mean=True, 
    plot_ts_median=True,
    ax=axes[0],
    title="CIRE before interpolation"
)

# 2. Plot ERA5 and NDVI Stats on the right
plot_variable_analysis(
    ds_intp, 
    variables_to_plot=["CIRE", "CIRE_min", "CIRE_max"],
    plot_ts_mean=True, 
    plot_ts_median=True,
    ax=axes[1],
    title="CIRE after interpolation"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# 1. Plot RGB on the left
plot_variable_analysis(
    ds, 
    variables_to_plot=["CIRE_min"], 
    plot_ts_mean=True, 
    ax=axes[0],
    title="CIRE before interpolation"
)

# 2. Plot ERA5 and NDVI Stats on the right
plot_variable_analysis(
    ds_intp, 
    variables_to_plot=["CIRE_min"],
    plot_ts_mean=True, 
    ax=axes[1],
    title="CIRE after interpolation"
)

plt.tight_layout()
plt.show()

In [ ]:
plot_variable_analysis(
    ds, 
    variables_to_plot=["vv_count"], 
    plot_ts_mean=True, 
    title="CIRE before interpolation"
)

In [ ]:
## Count und std auffüllen (eigtl nur bei vv und vh), da reindex:
# Max, Min wird interpoliert
count_vars = [v for v in ds.data_vars if "_count" in v or "_std"]

# NaNs in Counts sind faktisch 0 Beobachtungen
for v in count_vars:
    print(f"{v} values: {np.unique(ds[v].values)}")
    # ds[v] = ds[v].fillna(0).astype("uint8")

In [ ]:
plot_variable_analysis(
    ds_intp, 
    variables_to_plot=["vv", "vv_min", "vv_max"], 
    plot_ts_mean=True, 
    title="CIRE before interpolation"
)

In [ ]:
plot_variable_analysis(
    ds, 
    variables_to_plot=["IRECI"], 
    plot_ts_mean=True, 
    plot_ts_median=True # So siehst du den Unterschied zwischen "echt" und "wolken-gestört"
)

In [ ]:
plot_variable_analysis(ds, ["NDVI", "kNDVI", "CIRE", "IRECI", "NDMI"])

In [ ]:
plot_variable_analysis(ds, ["s2_final_mask_strict"])

In [ ]:
plot_variable_analysis(ds, ["vv", "vh", ])

In [ ]:
plot_variable_analysis(ds, ["s1_final_mask"])

In [ ]:
plot_variable_analysis(ds, ["pei_30_mean", "kNDVI"])

In [ ]:
plot_variable_analysis(ds, ["tp_rollingmax_mean", "kNDVI"])

In [ ]:
plot_variable_analysis(ds, ["t2mmax_max", "kNDVI"])

## Count maybe nur für einzelnen pixel

In [ ]:
plot_variable_analysis(ds, ["vv_count"])

In [ ]:
plot_variable_analysis(ds, ["NDVI_count"])

In [ ]:
plot_variable_analysis(ds, ["kNDVI_count"])